In [1]:
# imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Exploratory Data Analysis on GID Open Data

## Loading and Cleaning Data

In [2]:
# Defines relevant columns and scope of data
cols = [
    'service_request_id', 'service_request_parent_id', 'date_requested', 'case_age_days',
    'case_record_type', 'service_name', 'service_name_detail', 'date_closed', 'status', 
    'lat', 'lng', 'council_district', 'case_origin', 'referred', 'public_description'
    ]

years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026] #adjust years as needed

In [3]:
# Load datasets from url
for y in years:
    url = f'https://seshat.datasd.org/get_it_done_reports/get_it_done_requests_closed_{y}_datasd.csv'
    var = f'datasd_{y}'
    globals()[var] = pd.read_csv(url, usecols=cols)

C:\Users\cjjen\AppData\Local\Temp\ipykernel_14016\1403913400.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  globals()[var] = pd.read_csv(url, usecols=cols)


In [4]:
datasd = pd.concat([globals()[f'datasd_{y}'] for y in years], ignore_index=True)

In [5]:
datasd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3232200 entries, 0 to 3232199
Data columns (total 15 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   service_request_id         int64  
 1   service_request_parent_id  float64
 2   date_requested             object 
 3   case_age_days              float64
 4   case_record_type           object 
 5   service_name               object 
 6   service_name_detail        object 
 7   date_closed                object 
 8   status                     object 
 9   lat                        float64
 10  lng                        float64
 11  council_district           float64
 12  case_origin                object 
 13  referred                   object 
 14  public_description         object 
dtypes: float64(5), int64(1), object(9)
memory usage: 369.9+ MB


In [6]:
# Remove duplicate requests
datasd = datasd[datasd['service_request_parent_id'].isna()]

datasd.drop(columns=['service_request_parent_id'], inplace=True)

# Count total number of reports again from the deduplicated data
datasd['service_request_id'].nunique()

2811112

In [7]:
datasd[datasd.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id').head(16)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
1184017,3198116,2021-01-08 09:14:00.000,-999.0,ESD Complaint/Report,Environmental Services Code Compliance,Container Left Out,NaN,Closed,32.719486,-117.133231,3.0,Mobile,NaN,Not pick up yesterday
1285258,3198116,2021-01-08 09:14:00.000,NaN,ESD Complaint/Report,Environmental Services Code Compliance,Container Left Out,NaN,Closed,32.719486,-117.133231,3.0,Mobile,NaN,Not pick up yesterday
1285259,3201785,2021-01-11 20:33:00.000,NaN,ESD Complaint/Report,Parking Violation,Parking Zone Violation,NaN,Closed,32.752224,-117.135657,3.0,Mobile,NaN,Storage container taking up pretty valuable pa...
1213195,3201785,2021-01-11 20:33:00.000,-999.0,ESD Complaint/Report,Parking Violation,Parking Zone Violation,NaN,Closed,32.752224,-117.135657,3.0,Mobile,NaN,Storage container taking up pretty valuable pa...
1247567,3212991,2021-01-20 15:33:00.000,-999.0,ESD Complaint/Report,Trash/Recycling Collection,Out Late,NaN,Closed,32.703920,-117.097060,4.0,Phone,NaN,NaN
1285260,3212991,2021-01-20 15:33:00.000,NaN,ESD Complaint/Report,Trash/Recycling Collection,Out Late,NaN,Closed,32.703920,-117.097060,4.0,Phone,NaN,NaN
1285261,3263323,2021-03-03 09:37:00.000,NaN,ESD Complaint/Report,Missed Collection,Missed Collection,NaN,Closed,32.931340,-117.135360,6.0,Phone,NaN,Missed pick up
1143737,3263323,2021-03-03 09:37:00.000,-999.0,ESD Complaint/Report,Missed Collection,Missed Collection,NaN,Closed,32.931340,-117.135360,6.0,Phone,NaN,Missed pick up
1285262,3285114,2021-03-19 12:28:00.000,NaN,ESD Complaint/Report,Missed Collection,Missed Collection,NaN,Closed,32.754365,-117.071203,9.0,Mobile,NaN,Missed collection
1226132,3285114,2021-03-19 12:28:00.000,-999.0,ESD Complaint/Report,Missed Collection,Missed Collection,NaN,Closed,32.754365,-117.071203,9.0,Mobile,NaN,Missed collection


In [8]:
datasd_duplicates = datasd[datasd.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id')
datasd = datasd.drop_duplicates(subset=['service_request_id'], keep='last')

In [9]:
# Converting values to correct dtypes
datasd['date_requested'] = pd.to_datetime(datasd['date_requested'])
datasd['date_closed'] = pd.to_datetime(datasd['date_closed'])
datasd['council_district'] = datasd['council_district'].astype('category')

In [10]:
datasd.loc[datasd['referred'].notna(), 'referred'].head()

0     This report has been referred to Caltrans San ...
5     This report has been referred to Caltrans San ...
25    This report has been referred to Other departm...
32    This report has been referred to Police Depart...
33    This report has been referred to Police Depart...
Name: referred, dtype: object

In [11]:
# Inplace function(s) and code to clean up referred column. Based on the frequency of referrals made
def categorize_referrals(txt):
    txt = str(txt).lower()
    if 'sdge' in txt or 'sdg&e' in txt:
        return 'sdg&e'
    if 'cox.com' in txt:
        return 'cox communications'
    if 'charter.com' in txt or 'spectrum' in txt:
        return 'spectrum'
    if 'att' in txt or 'at&t' in txt:
        return 'at&t'
    if 'cleanharbors' in txt:
        return 'clean harbors sanitation'
    if 'wm.com' in txt or 'waste management' in txt:
        return 'waste management'
    if 'republicservices' in txt or 'republic-services' in txt:
        return 'republic services'
    if 'edcodisposal' in txt:
        return 'edco disposal'
    if 'sdmts' in txt:
        return 'mts'
    if 'nctd.org' in txt:
        return 'nctd'
    if 'bnsf' in txt:
        return 'bnsf railroad'
    if 'caltrans' in txt or 'dot.ca.gov' in txt:
        return 'caltrans'
    if 'portofsandiego' in txt or 'port of san diego' in txt:
        return 'port of sd'
    if 'san.org' in txt:
        return 'sd international airport'
    if 'sandi.net' in txt:
        return 'sd unified school district'
    if 'ucsd.edu' in txt:
        return 'ucsd'
    if 'sdsu.edu' in txt:
        return 'sdsu'
    if 'sdccd.edu' in txt:
        return 'sdccd'
    if 'chulavistaca.gov' in txt:
        return 'city of chula vista'
    if 'sdcounty.ca.gov' in txt:
        return 'san diego county'
    if 'parks.ca.gov' in txt or 'wildlife.ca.gov' in txt:
        return 'california state parks/wildlife'
    if 'usps' in txt:
        return 'usps'
    if 'livelmh' in txt:
        return 'liberty military housing'
    if 'improvedtsd' in txt or 'ballparkdistrict' in txt or 'downtownsandiego.org' in txt:
        return 'downtown clean & safe / ballpark'
    if 'sdhumane' in txt:
        return 'sd humane society'
    if 'vector' in txt:
        return 'county vector control'
    if 'rms' in txt or 'retailsolutions' in txt or 'retailms.net' in txt:
        return 'retail solutions (rms)'
    if 'smartcart' in txt:
        return 'smart cart'
    if 'clearchannel' in txt:
        return 'clear channel'
    if 'outfront.com' in txt:
        return 'outfront media'
    if 'engineering' in txt:
        return 'field engineering'
    if 'facilitiesmaint' in txt:
        return 'facilities maintenance'
    if 'treepermit' in txt:
        return 'tree permits'
    if 'publicliability' in txt or 'riskmanagement' in txt:
        return 'risk management / public liability'
    if 'specialevents' in txt:
        return 'special events'
    if 'realestate' in txt:
        return 'real estate assets'
    if 'sdpd' in txt or 'police' in txt:
        return 'sdpd'
    if 'sdfd' in txt or 'fire' in txt:
        return 'sdfd'
    if 'wcm_gid' in txt or 'water' in txt:
        return 'water / wastewater (pud)'
    if 'jkafulides' in txt or 'parking' in txt:
        return 'parking meter repair'
    if 'sdlg' in txt or 'lifeguard' in txt:
        return 'lifeguards'
    if 'fit_dsd' in txt or 'development services' in txt:
        return 'development services (dsd)'
    if 'sdbusiness' in txt:
        return 'sd business / economic dev'
    if 'street division' in txt:
        return 'street division'
    if 't-row code compliance' in txt:
        return 't-row code compliance'     
    if 'sanitization contractor' in txt or 'environmental services' in txt or 'esd' in txt:
        return 'environmental services (esd)'
    if 'parks intake' in txt:
        return 'parks & recreation'
    if 'testing only' in txt:
        return 'test/invalid'
    if 'null' in txt:
        return 'error'
    return txt

def clean_referrals(df):
    df['referred'] = df['referred'].apply(categorize_referrals)
    # If the referral is not a specific organization (is an email), extract the email:
    df.loc[df['referred'].str.contains('@'), 'referred'] = df.loc[df['referred'].str.contains('@'), 'referred'].str.extract(r'([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})')[0].str.lower()
    df.loc[df['referred'] == 'nan', 'referred'] = np.nan

In [12]:
clean_referrals(datasd)

datasd.loc[datasd['referred'].notna(), 'referred'].head()

0                 caltrans
5                 caltrans
25    gbowers@sandiego.gov
32                    sdpd
33                    sdpd
Name: referred, dtype: object

In [13]:
datasd.shape

(2811112, 14)

In [14]:
datasd.head()

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
0,100848,2016-08-21 10:42:00,-999.0,TSW,Traffic Signal Issue,NaN,NaT,Referred,33.248617,-117.274685,NaN,Web,caltrans,I drove thru about 10am this morning the 21st....
1,100916,2016-08-21 19:22:00,54.0,TSW,Other,NaN,2016-10-14,Closed,NaN,NaN,NaN,Email,NaN,I was wondering when this might be addressed.\...
2,102295,2016-08-29 17:01:00,1.0,TSW,Traffic Signal Issue,NaN,2016-08-30,Closed,33.018313,-117.115255,NaN,Web,NaN,The light at the intersection on the south eas...
3,103729,2016-09-06 10:40:00,22.0,Traffic Engineering,Traffic Engineering,NaN,2016-09-29,Closed,NaN,NaN,NaN,Phone,NaN,"We own the property at 4626 Mercury, the stree..."
4,104040,2016-09-07 13:25:00,6.0,TSW,Pavement Maintenance,NaN,2016-09-14,Closed,NaN,NaN,NaN,Phone,NaN,very dangerous dip - East side of intersection...


In [15]:
gid_open = pd.read_csv('https://seshat.datasd.org/get_it_done_reports/get_it_done_requests_open_datasd.csv', usecols=cols)

# Remove duplicate requests
gid_open = gid_open[gid_open['service_request_parent_id'].isna()]

gid_open.drop(columns=['service_request_parent_id'], inplace=True)

In [16]:
gid_open[gid_open.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id').head(10)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description


In [17]:
gid_open['date_requested'] = pd.to_datetime(gid_open['date_requested'])
gid_open['council_district'] = gid_open['council_district'].astype('category')
clean_referrals(gid_open)

In [18]:
gid_open.shape

(62875, 14)

In [19]:
gid_open.head()

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
0,5460714,2025-11-21 08:34:41,349,TSW ROW,Graffiti - Code Enforcement,Other,NaN,New,32.804304,-117.154995,7.0,Web,t-row code compliance,"Graffiti on the west side of the 163, it is fa..."
1,5460909,2025-11-21 10:04:12,349,TSW,Street Light Maintenance,STREET LIGHT OUT,NaN,In Process,32.709911,-117.136301,8.0,Mobile,NaN,Lights not turning on at basketball court. Dar...
2,5460966,2025-11-21 10:24:08,349,TSW,Pavement Maintenance,MILL & PAVE TRENCH REPAIRS,NaN,In Process,32.919324,-117.227482,1.0,Phone,NaN,CONS REPORTS BROKEN MONUMENT LID
4,5461169,2025-11-21 11:51:39,349,TSW ROW,Graffiti - Code Enforcement,Commercial Property,NaN,In Process,32.719586,-117.166116,3.0,Mobile,NaN,Requesting assistance in removal of graffiti i...
5,5461751,2025-11-21 17:27:11,349,TSW,Street Light Maintenance,STREET LIGHT OUT,NaN,In Process,32.723212,-117.165366,3.0,Web,NaN,The two street lights that are inside the park...


In [20]:
df = pd.concat([datasd, gid_open], ignore_index=True)

C:\Users\cjjen\AppData\Local\Temp\ipykernel_14016\4261117055.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([datasd, gid_open], ignore_index=True)


In [21]:
df[df.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id').head(10)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
320876,388835,2018-08-06 15:34:00,476.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2019-11-25,Closed,NaN,NaN,NaN,NaN,NaN,TEMPORARY TRENCHES / OPEN POTHOLES WITH NO PER...
2858012,388835,2018-08-06 15:34:11,3013.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,NaN,NaN,NaN,NaN,NaN,NaN
2846013,2408745,2018-09-25 14:46:37,2963.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,32.754192,-117.077914,9.0,NaN,NaN,NaN
410249,2408745,2018-09-25 14:46:00,224.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2019-05-07,Closed,32.754192,-117.077914,9.0,NaN,NaN,TRENCH PLATES MORE THAN 30 DAYS- CORNERS OF TR...
2846019,2410615,2018-09-27 14:15:30,2961.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,40.684326,-73.995116,NaN,NaN,NaN,NaN
320630,2410615,2018-09-27 14:15:00,293.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2019-07-17,Closed,40.684326,-73.995116,NaN,NaN,NaN,CROWN CASTLE FAILED TO RESTORE TRENCHES AFTER ...
417164,2410670,2018-09-27 14:53:00,293.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2019-07-17,Closed,32.722485,-117.164600,3.0,NaN,NaN,TRENCHES NEVER RESTORED; ISSUED NOV ON 5/7/2018-
2846020,2410670,2018-09-27 14:53:12,2961.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,32.722485,-117.164600,3.0,NaN,NaN,NaN
2846043,2413618,2018-10-02 07:23:46,2956.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,32.914834,-117.081643,5.0,NaN,NaN,NaN
917899,2413618,2018-10-02 07:23:00,1016.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2021-07-14,Closed,32.914834,-117.081643,5.0,NaN,NaN,TRENCH WAS NEVER RESTORED- NO PERMIT


In [22]:
df_duplicates = df[df.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id')
df = df.drop_duplicates(subset=['service_request_id'], keep='last')

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2873883 entries, 0 to 2873986
Data columns (total 14 columns):
 #   Column               Dtype         
---  ------               -----         
 0   service_request_id   int64         
 1   date_requested       datetime64[ns]
 2   case_age_days        float64       
 3   case_record_type     object        
 4   service_name         object        
 5   service_name_detail  object        
 6   date_closed          datetime64[ns]
 7   status               object        
 8   lat                  float64       
 9   lng                  float64       
 10  council_district     category      
 11  case_origin          object        
 12  referred             object        
 13  public_description   object        
dtypes: category(1), datetime64[ns](2), float64(3), int64(1), object(7)
memory usage: 309.7+ MB


In [24]:
df.head()

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
0,100848,2016-08-21 10:42:00,-999.0,TSW,Traffic Signal Issue,NaN,NaT,Referred,33.248617,-117.274685,NaN,Web,caltrans,I drove thru about 10am this morning the 21st....
1,100916,2016-08-21 19:22:00,54.0,TSW,Other,NaN,2016-10-14,Closed,NaN,NaN,NaN,Email,NaN,I was wondering when this might be addressed.\...
2,102295,2016-08-29 17:01:00,1.0,TSW,Traffic Signal Issue,NaN,2016-08-30,Closed,33.018313,-117.115255,NaN,Web,NaN,The light at the intersection on the south eas...
3,103729,2016-09-06 10:40:00,22.0,Traffic Engineering,Traffic Engineering,NaN,2016-09-29,Closed,NaN,NaN,NaN,Phone,NaN,"We own the property at 4626 Mercury, the stree..."
4,104040,2016-09-07 13:25:00,6.0,TSW,Pavement Maintenance,NaN,2016-09-14,Closed,NaN,NaN,NaN,Phone,NaN,very dangerous dip - East side of intersection...


## EDA

### What is the current workload and system?

In [25]:
N = 15 # global variable

In [26]:
complete_yrs = df[df['date_requested'].dt.year >= min(years)].copy()
print('Total Number of Distinct Services Offered Every Year')
complete_yrs['year_opened'] = complete_yrs['date_requested'].dt.year
complete_yrs.groupby('year_opened')[['service_name', 'service_name_detail']].nunique()

Total Number of Distinct Services Offered Every Year


,service_name,service_name_detail
year_opened,,
2016,23,63
2017,26,124
2018,32,187
2019,31,159
2020,34,154
2021,37,163
2022,36,199
2023,40,217
2024,40,253


In [27]:
intake = complete_yrs.groupby(complete_yrs['date_requested'].dt.to_period('M')).size().reset_index(name='intake')
intake['month'] = intake['date_requested'].dt.to_timestamp()

outflow = complete_yrs.groupby(complete_yrs['date_closed'].dt.to_period('M')).size().reset_index(name='outflow')
outflow['month'] = outflow['date_closed'].dt.to_timestamp()

req_vol_df = pd.merge(intake[['month', 'intake']], outflow[['month', 'outflow']], on='month', how='outer').fillna(0)
req_vol_df.sort_values('month', inplace=True)

# Filter out incomplete month (October 2026)
req_vol_df = req_vol_df[req_vol_df['month'] < '2026-10-01']

In [28]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=req_vol_df['month'], y=req_vol_df['intake'], mode='lines', name='Reports Received'))
fig.add_trace(go.Scatter(x=req_vol_df['month'], y=req_vol_df['outflow'], mode='lines', name='Reports Completed'))
fig.update_layout(
    title=f"Monthly Intake and Outflow ({min(years)}-{max(years)})<br><span style='font-size: 13px; color: gray;'>Outflow does not necessarily mean any work occurred.</span>",
    xaxis_title='Date (Month Yr.)',
    yaxis_title='Total Reports',
    hovermode='x unified',
    xaxis={'tickformat': '%b %Y', 'hoverformat': '%b %Y'}
    )

In [29]:
referrals = datasd\
    .groupby('service_name')['status']\
    .apply(
        lambda t: (t == 'Referred').mean().round(4) * 100
        )\
    .reset_index(name='proportion')\
    .sort_values('proportion', ascending=False)
px.bar(
    referrals[:N],
    x='service_name',
    y='proportion',
    title=f'Highest {N} Proportion of Referrals by Service ({min(years)}-{max(years)})',
    labels={'service_name': 'Service Category', 'proportion': 'Proportion'}
    )

In [30]:
med_rsps_days = datasd.groupby('service_name')['case_age_days'].median()
highest_med_days = med_rsps_days.sort_values(ascending=False).reset_index()[:N]
fig = px.bar(
    highest_med_days.sort_values('case_age_days'),
    y='service_name',
    x='case_age_days',
    text_auto='.f',
    title=f'Top {N} Services With Highest Median Time (Days) to Respond',
    labels={'case_age_days': 'Median Response Time (Days)', 'service_name': 'Service Category'}
    )
fig.update_traces(hoverinfo='none', hovertemplate=None, textposition='outside')

In [31]:
med_rsps_days = datasd.groupby('service_name')['case_age_days'].median()
lowest_med_days = med_rsps_days.sort_values().reset_index()[:N]
fig = px.bar(lowest_med_days.sort_values('case_age_days', ascending=False), y='service_name', x='case_age_days', text_auto='.f')
fig.update_traces(hoverinfo='none', hovertemplate=None, textposition='outside')

Given that `datasd` contains closed cases, these visuals show what the City prioritizes (and responds to efficiently) and where responses might be delayed. Notably, there is a common theme among services with high median case ages: private property or street/traffic-related services.

Does `case_age_days` differ based on `service`?

In [32]:
from scipy.stats import kruskal

# Use closed reports with valid case durations
kruskalwallis_data = datasd[
    datasd['service_name'].notna()
    & datasd['case_age_days'].notna()
    & (datasd['case_age_days'] >= 0)
].copy()

# Select services with sufficient observations
top_services = (
    kruskalwallis_data['service_name']
    .value_counts()
    .loc[lambda counts: counts > 30]
    .index
)

groups = [
    kruskalwallis_data.loc[
        kruskalwallis_data['service_name'] == service,
        'case_age_days'
    ].to_numpy()
    for service in top_services
    ]

print(f'Number of distinct services kept: {top_services.nunique()}\n')

statistic, p_value = kruskal(*groups)

print(f'Kruskal-Wallis statistic: {statistic:.3f}')
print(f'p-value: {p_value:.4e}')

Number of distinct services kept: 42

Kruskal-Wallis statistic: 864284.834
p-value: 0.0000e+00


Is `referred` dependent of `service` category?

In [33]:
from scipy.stats import chi2_contingency

chi2_data = datasd[datasd['service_name'].notna()].copy()
chi2_data['is_referred'] = chi2_data['referred'].notna()

contingency_table = pd.crosstab(chi2_data['service_name'], chi2_data['is_referred'])

print(f'First 5 rows of contingency table:\n{contingency_table.head()}\n')

chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)

print(f'Chi-square Statistic: {chi2_stat:.2f}')
print(f'P-value: {p_val:.4e}')

First 5 rows of contingency table:
is_referred                        False  True 
service_name                                   
ADA Compliance                         1      0
Bicycle-Related Issue                  4      1
Biohazard                              7    480
Building and Land Use Enforcement   1445    556
COVID-19                               5    967

Chi-square Statistic: 814231.43
P-value: 0.0000e+00


### Where might a computer vision model improve the current system?
<!-- 2. What is the current workload and system? -->

In [34]:
# What is the most common service still unaddressed from each year? (as of October 2026)
open_servs = pd.crosstab(index=gid_open['service_name'], columns=gid_open['date_requested'].dt.year).idxmax()
open_servs 

date_requested
2016        Pavement Maintenance
2017       Sidewalk Repair Issue
2018       Sidewalk Repair Issue
2019       Sidewalk Repair Issue
2020       Sidewalk Repair Issue
2021       Sidewalk Repair Issue
2022       Sidewalk Repair Issue
2023       Sidewalk Repair Issue
2024       Sidewalk Repair Issue
2025    Street Light Maintenance
2026          Parking - 72-Hours
dtype: object

In [35]:
open_reports = gid_open.copy()
open_reports['year'] = open_reports['date_requested'].dt.year
open_reports = open_reports[open_reports['year'] != 2026]
open_reports_data = open_reports.groupby(['year', 'service_name']).size().reset_index(name='count')
px.bar(
    open_reports_data[open_reports_data['service_name'].isin(open_reports['service_name'].value_counts().index[:5])],
    x='year',
    y='count',
    color='service_name',
    barmode='group',
    title=f"Breakdown of Most Common Open Reports by Year Originally Reported",
    labels={'year': 'Year Opened', 'service_name': 'Service Category', 'count': 'Total Reports'},
    )

`Sidewalk Repair Issue` cases are consistenly the largest amount of open reports in earlier years, followed by `Pavement Maintenance`.


<!-- from docx import Document
doc = Document('ROW Support Matrix.docx').tables[0]

data = [[cell.text.strip() for cell in row.cells] for row in doc.rows]

matrix = pd.DataFrame(data[1:], columns=data[0])
matrix = matrix.replace('', np.nan) -->

From before, we saw that `Other` had the second highest proportion of referrals (after `Biohazard`). The most common referral is SDPD (San Diego Police Department); The average response time for these referrals is 0.00

In [36]:
datasd.loc[datasd['service_name'] == 'Other', 'service_name_detail'].unique()

array([nan], dtype=object)

In [37]:
other_referrals = datasd.loc[(datasd['service_name'] == 'Other'), 'referred'].value_counts()
px.bar(other_referrals[:N])

In [38]:
px.box(datasd.loc[(datasd['service_name'] == 'Other') & (datasd['referred'].notna()) & (datasd['referred'].isin(other_referrals[:N].index)), 'case_age_days'], orientation='h')

**The existing Get It Done system processes a substantial volume of public requests, but the available open data does not establish how accurately or efficiently requests are initially routed. My understanding is that City intake staff manually use the ROW Matrix Service to help determine the appropriate service category. An image-assisted classification or validation tool could complement this workflow by providing additional evidence from submitted images, descriptions, and location data when assigning service categories.**

### What can be measured operationally?

This data does not provide enough information about how misrouted cases are dealt with, such as the timeframe between the initial routing and the correction.

What we currently have is the number of days taken for a case's status to update, i.e. from `Open` to `In Process`, and even then it only contains the most recent status update. So while `case_age_days` can tell us how long the process takes for cases and if there are any systemic problems like certain services seeing overly delayed responses, we cannot measure the operational impacts at least in terms of delays.
 
Lastly, the duplicated cases from before indicate some intervention, like updating a service category. But these appear to be errors when updating/transfering the data to the open portal.

In [39]:
datasd_duplicates.head(16)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
1184017,3198116,2021-01-08 09:14:00.000,-999.0,ESD Complaint/Report,Environmental Services Code Compliance,Container Left Out,NaN,Closed,32.719486,-117.133231,3.0,Mobile,NaN,Not pick up yesterday
1285258,3198116,2021-01-08 09:14:00.000,NaN,ESD Complaint/Report,Environmental Services Code Compliance,Container Left Out,NaN,Closed,32.719486,-117.133231,3.0,Mobile,NaN,Not pick up yesterday
1285259,3201785,2021-01-11 20:33:00.000,NaN,ESD Complaint/Report,Parking Violation,Parking Zone Violation,NaN,Closed,32.752224,-117.135657,3.0,Mobile,NaN,Storage container taking up pretty valuable pa...
1213195,3201785,2021-01-11 20:33:00.000,-999.0,ESD Complaint/Report,Parking Violation,Parking Zone Violation,NaN,Closed,32.752224,-117.135657,3.0,Mobile,NaN,Storage container taking up pretty valuable pa...
1247567,3212991,2021-01-20 15:33:00.000,-999.0,ESD Complaint/Report,Trash/Recycling Collection,Out Late,NaN,Closed,32.703920,-117.097060,4.0,Phone,NaN,NaN
1285260,3212991,2021-01-20 15:33:00.000,NaN,ESD Complaint/Report,Trash/Recycling Collection,Out Late,NaN,Closed,32.703920,-117.097060,4.0,Phone,NaN,NaN
1285261,3263323,2021-03-03 09:37:00.000,NaN,ESD Complaint/Report,Missed Collection,Missed Collection,NaN,Closed,32.931340,-117.135360,6.0,Phone,NaN,Missed pick up
1143737,3263323,2021-03-03 09:37:00.000,-999.0,ESD Complaint/Report,Missed Collection,Missed Collection,NaN,Closed,32.931340,-117.135360,6.0,Phone,NaN,Missed pick up
1285262,3285114,2021-03-19 12:28:00.000,NaN,ESD Complaint/Report,Missed Collection,Missed Collection,NaN,Closed,32.754365,-117.071203,9.0,Mobile,NaN,Missed collection
1226132,3285114,2021-03-19 12:28:00.000,-999.0,ESD Complaint/Report,Missed Collection,Missed Collection,NaN,Closed,32.754365,-117.071203,9.0,Mobile,NaN,Missed collection


In [40]:
df_duplicates.head(16)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
320876,388835,2018-08-06 15:34:00,476.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2019-11-25,Closed,NaN,NaN,NaN,NaN,NaN,TEMPORARY TRENCHES / OPEN POTHOLES WITH NO PER...
2858012,388835,2018-08-06 15:34:11,3013.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,NaN,NaN,NaN,NaN,NaN,NaN
2846013,2408745,2018-09-25 14:46:37,2963.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,32.754192,-117.077914,9.0,NaN,NaN,NaN
410249,2408745,2018-09-25 14:46:00,224.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2019-05-07,Closed,32.754192,-117.077914,9.0,NaN,NaN,TRENCH PLATES MORE THAN 30 DAYS- CORNERS OF TR...
2846019,2410615,2018-09-27 14:15:30,2961.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,40.684326,-73.995116,NaN,NaN,NaN,NaN
320630,2410615,2018-09-27 14:15:00,293.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2019-07-17,Closed,40.684326,-73.995116,NaN,NaN,NaN,CROWN CASTLE FAILED TO RESTORE TRENCHES AFTER ...
417164,2410670,2018-09-27 14:53:00,293.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2019-07-17,Closed,32.722485,-117.164600,3.0,NaN,NaN,TRENCHES NEVER RESTORED; ISSUED NOV ON 5/7/2018-
2846020,2410670,2018-09-27 14:53:12,2961.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,32.722485,-117.164600,3.0,NaN,NaN,NaN
2846043,2413618,2018-10-02 07:23:46,2956.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,32.914834,-117.081643,5.0,NaN,NaN,NaN
917899,2413618,2018-10-02 07:23:00,1016.0,TSW ROW,Right-of-Way Code Enforcement,Trenches,2021-07-14,Closed,32.914834,-117.081643,5.0,NaN,NaN,TRENCH WAS NEVER RESTORED- NO PERMIT


# How many similar instances of case closed same day exist in your dataset, and are these anomalies?

In [41]:
df['date_requested'] = pd.to_datetime(df['date_requested'])
df['date_closed'] = pd.to_datetime(df['date_closed'])

same_day = df['date_closed'].dt.normalize() == df['date_requested'].dt.normalize()
print(f'{same_day.sum():,} of {len(df):,} total cases = {same_day.mean():.2%}')

779,141 of 2,873,883 total cases = 27.11%


In [42]:
import pandas as pd

d = df.copy()
d['date_requested'] = pd.to_datetime(d['date_requested'], errors='coerce')
d['date_closed'] = pd.to_datetime(d['date_closed'], errors='coerce')

closed = d[d['date_closed'].notna() & d['date_requested'].notna()].copy()
closed['age_calc'] = (closed['date_closed'].dt.normalize() - closed['date_requested'].dt.normalize()).dt.days
closed['same_day'] = closed['age_calc'] == 0

# Use the closed DataFrame, which contains one row per closed case
# and a Boolean `same_day` column.

TOP_N = 15

# Remove rows without a service category
data = closed.dropna(subset=['service_name']).copy()

# Create outcome: closed same day or not
data['outcome'] = np.where(
    data['same_day'],
    'Closed same day',
    'Not closed same day'
)

# Find the 15 service categories with the most closed cases
top_categories = (
    data['service_name']
    .value_counts()
    .head(TOP_N)
    .index
)

data = data[data['service_name'].isin(top_categories)]

# Count same-day and non-same-day closures by service category
pv = (
    data.groupby(['service_name', 'outcome'])
    .size()
    .unstack(fill_value=0)
)

# Make sure both outcome columns exist
for col in ['Closed same day', 'Not closed same day']:
    if col not in pv.columns:
        pv[col] = 0

# Sort categories by total volume
totals = pv.sum(axis=1)
order = totals.sort_values().index.tolist()

# Calculate percentages within each service category
pct = pv.div(totals, axis=0) * 100

# Colors
COLORS = {
    'Closed same day': '#008F8A',
    'Not closed same day': '#E0A458'
}

def k(n):
    return f'{n:,.0f}' if n < 1000 else f'{n/1000:,.0f}K'

fig = go.Figure()

# Create stacked bars
for col in ['Closed same day', 'Not closed same day']:
    fig.add_bar(
        y=order,
        x=[pct.loc[d, col] for d in order],
        orientation='h',
        name=col,
        marker_color=COLORS[col],
        customdata=[
            [pv.loc[d, col], totals[d]]
            for d in order
        ],
        hovertemplate=(
            '%{y}<br>'
            + col + ': %{x:.1f}% '
            '(%{customdata[0]:,} cases)<br>'
            'Total closed cases: %{customdata[1]:,}'
            '<extra></extra>'
        )
    )

# Add total case counts beside each bar
fig.add_scatter(
    y=order,
    x=[103] * len(order),
    mode='text',
    showlegend=False,
    hoverinfo='skip',
    text=[f'({k(totals[d])} cases)' for d in order],
    textposition='middle right'
)

# Axis formatting
fig.update_yaxes(
    categoryorder='array',
    categoryarray=order,
    title=''
)

fig.update_xaxes(
    range=[0, 120],
    title='% of closed cases within each service category',
    ticksuffix='%',
    showgrid=False
)

# Layout
fig.update_layout(
    barmode='stack',
    plot_bgcolor='white',
    height=560,
    margin=dict(l=10, r=40, t=100, b=60),
    legend=dict(
        orientation='h',
        y=-0.15,
        title=''
    ),
    title=dict(
        text=(
            f'Top {TOP_N} service categories by closed-case volume'
            '<br><sup>Percentage of cases closed on the same day vs. '
            'not closed on the same day; totals shown beside each bar.</sup>'
        )
    )
)

fig.show()